In [ ]:
# TPU VM host as a CPU-only wmr machine: hardware, then 072q-1 (only 'Huohua Studio', checked) CPU-only:
# one segment alone (profiled), then all segments in parallel processes. Only text leaves Kaggle.
import glob, os, shutil, subprocess, time, zipfile
VERSION, DATASET, W = '0.5.4', 'huohua-studio-test-data', '/kaggle/working'
R = open(f'{W}/results.txt', 'w')
def say(s):
    print(s, flush=True); R.write(s + '\n'); R.flush()
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
say('nproc ' + sh('nproc') + ', os.cpu_count ' + str(os.cpu_count()))
say(sh("lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread|Core|Socket|MHz|NUMA node\\(s\\)|L3'"))
say(sh('free -g | head -2'))
say('python ' + sh('python --version') + '; ' + sh("pip list 2>/dev/null | grep -iE '^(torch|torch-xla|jax|opencv|av|onnxruntime|numpy) '"))
say('ffmpeg: ' + (shutil.which('ffmpeg') or 'none'))
z = glob.glob(f'/kaggle/input/**/wmr-{VERSION}.zip', recursive=True)
d = glob.glob(f'/kaggle/input/**/wmr-{VERSION}/wmr/track.py', recursive=True)
if z:
    zipfile.ZipFile(z[0]).extractall(W)
else:   # Kaggle unpacks zips when it mounts a dataset
    shutil.copytree(os.path.dirname(os.path.dirname(d[0])), f'{W}/wmr-{VERSION}', dirs_exist_ok=True)
IN_DIR = next(b for b, d, f in os.walk('/kaggle/input', followlinks=True)
              if DATASET in b and any(x.lower().endswith('.mp4') for x in f))


In [ ]:
# CPU setup: no GPU build of ONNX Runtime, no cuDNN
P = f'{W}/wmr-{VERSION}'
t = time.time()
say(sh(f'pip install -q -r {P}/requirements/kaggle.txt onnxruntime imageio-ffmpeg 2>&1 | tail -2'))
say(sh('python -c "import cv2" 2>&1 || pip install -q opencv-python-headless 2>&1 | tail -1'))
say(sh(f'pip install -q --no-deps {P} 2>&1 | tail -1'))
say(sh('python -m wmr doctor 2>&1 | tail -12'))
FF = shutil.which('ffmpeg') or sh('python -c "import imageio_ffmpeg as f; print(f.get_ffmpeg_exe())"')
say(f'setup {time.time() - t:.0f} s; ffmpeg {FF}')


In [ ]:
# Model download once (parallel workers would race), then cut 072q-1 into K segments at keyframes
say(sh('python -c "from wmr.detect import OCR; o = OCR(use_gpu=False); print(o.model, o.provider)" 2>&1 | tail -1'))
N = os.cpu_count()
K = max(2, min(32, N // 4))
TH = max(1, N // K)
src = os.path.join(IN_DIR, '072q-1.watermarked.mp4')
SEG = f'{W}/seg'
os.makedirs(SEG, exist_ok=True)
dur = 942.0
sh(f'{FF} -v error -i "{src}" -map 0 -c copy -f segment -segment_time {dur / K:.2f} -reset_timestamps 1 {SEG}/s%03d.mp4')
segs = sorted(os.listdir(SEG))
say(f'K {K} segments ({len(segs)} made), {TH} threads each')
open(f'{W}/texts.txt', 'w').write('Huohua Studio\n')
KEYS = ('[analyze] pass', '[remove] ', '[verify]', '[run] seconds', 'Error', 'Traceback')
def run(seg, env, out):
    return subprocess.Popen(['python', '-m', 'wmr', 'run', f'{SEG}/{seg}', '-o', out, '--texts', f'{W}/texts.txt', '--cpu'],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, **env})


In [ ]:
# A: segment 0 alone, all cores, profiled
t = time.time()
p = run(segs[0], {'WMR_PROF': f'{W}/prof.txt'}, f'{W}/a.mp4')
o = p.communicate()[0]
say(f'=== A: {segs[0]} alone: {time.time() - t:.0f} s, exit {p.returncode}')
for l in o.splitlines():
    if any(k in l for k in KEYS):
        say('  ' + l.strip()[:300])


In [ ]:
# B: all segments at once, one process each, WMR_THREADS per process
os.makedirs(f'{W}/b', exist_ok=True)
t = time.time()
ps = [(s, run(s, {'WMR_THREADS': str(TH)}, f'{W}/b/{s}')) for s in segs]
outs = [(s, p.communicate()[0], p.returncode) for s, p in ps]
say(f'=== B: {len(segs)} segments in parallel: {time.time() - t:.0f} s wall for 942 s of video'
    f' (T4 x2 run: 2199 s), exits {sorted(set(r for _, _, r in outs))}')
leak = chk = 0
import re
for s, o, r in outs:
    for l in o.splitlines():
        if '[run] seconds' in l or 'Error' in l or 'Traceback' in l:
            say(f'  {s}: ' + l.strip()[:250])
        m = re.search(r'\[verify\] (\d+) of (\d+)', l)
        if m:
            leak += int(m.group(1)); chk += int(m.group(2))
say(f'  verify total: {leak} of {chk} sampled frames still show expected text (T4 run: 12 of 1884)')
R.close()


In [ ]:
# keep only text
for x in os.listdir(W):
    if x not in ('results.txt', 'prof.txt'):
        p = os.path.join(W, x); shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
!ls -la /kaggle/working
